In [15]:
from tdc.single_pred import Tox
import pandas as pd, numpy as np

data  = Tox(name='herg_karim')
df    = data.get_data()
split = data.get_split()   

Found local copy...
Loading...
Done!


In [17]:
import os, json, hashlib, datetime
import numpy as np, pandas as pd
from rdkit import Chem
from rdkit import RDLogger; RDLogger.DisableLog('rdApp.*')

NEW  = 'C:/Users/iisl/Desktop/0_Molecualar_Informatics/2026_final/2026_new/'
PREP = NEW + '00_data-preprocess/'
RAW  = PREP + 'data/herg_karim.tab'
OUT  = NEW + 'data/processed/herg_karim/'
os.makedirs(OUT, exist_ok=True)

try:
    from importlib.metadata import version as _ver
    PYTDC_VERSION = _ver('PyTDC')
except Exception:
    import tdc; PYTDC_VERSION = getattr(tdc, '__version__', 'unknown')

print('PyTDC   :', PYTDC_VERSION)
print('raw 있음:', os.path.exists(RAW), '|', RAW)

PyTDC   : 1.1.15
raw 있음: True | C:/Users/iisl/Desktop/0_Molecualar_Informatics/2026_final/2026_new/00_data-preprocess/data/herg_karim.tab


In [18]:
from tdc.single_pred import Tox

data  = Tox(name='herg_karim', path=PREP + 'data')
df    = data.get_data()
split = data.get_split()        # 기본 random, seed 42, frac [0.7, 0.1, 0.2]

print('=== 기본 ===')
print('shape :', df.shape)
print('cols  :', list(df.columns))
print('split :', {k: len(v) for k, v in split.items()},
      '| 합계', sum(len(v) for v in split.values()))

print('\n=== 라벨 Y ===')
print('unique 개수 :', df['Y'].nunique())
print('unique 값   :', sorted(df['Y'].unique())[:10])
print('분포        :', df['Y'].value_counts().to_dict())
print('→ 이진 라벨인가?', df['Y'].nunique() == 2)

print('\n=== 결측 / 중복 ===')
print('결측        :', df.isna().sum().to_dict())
print('Drug_ID 중복:', int(df['Drug_ID'].duplicated().sum()))
print('SMILES 중복 :', int(df['Drug'].duplicated().sum()))
df.head()

Found local copy...
Loading...
Done!


=== 기본 ===
shape : (13445, 3)
cols  : ['Drug_ID', 'Drug', 'Y']
split : {'train': 9412, 'valid': 1344, 'test': 2689} | 합계 13445

=== 라벨 Y ===
unique 개수 : 2
unique 값   : [0, 1]
분포        : {0: 6727, 1: 6718}
→ 이진 라벨인가? True

=== 결측 / 중복 ===
결측        : {'Drug_ID': 0, 'Drug': 0, 'Y': 0}
Drug_ID 중복: 0
SMILES 중복 : 0


,Drug_ID,Drug,Y
0,0,Fc1ccc(-n2cc(NCCN3CCCCC3)nn2)cc1F,1
1,1,COc1cc(N2Cc3ccc(Sc4ccc(F)cc4)nc3C2=O)ccc1OCCN1...,0
2,2,CCOC(=O)[C@H]1CC[C@@H](N2CC(NC(=O)CNc3nn(C(N)=...,0
3,3,N[C@@H](Cn1c(=O)cnc2ccc(F)cc21)C1CCC(NCc2ccc3c...,0
4,4,O=C(NC1COc2cccc(-c3ccnc(CO)c3)c2C1)c1ccc(OCC(F...,0


In [19]:
def canon(smi):
    m = Chem.MolFromSmiles(str(smi))
    if m is None or m.GetNumAtoms() == 0 or m.GetNumBonds() == 0:
        return None, None, None
    return (Chem.MolToSmiles(m),
            Chem.MolToInchiKey(m).split('-')[0],
            m.GetNumAtoms())

rec = [canon(s) for s in df['Drug']]          # 13,445개 → 1~2분
df['canon_smiles'] = [r[0] for r in rec]
df['key14']        = [r[1] for r in rec]
df['n_atoms']      = [r[2] for r in rec]

bad = df['key14'].isna()
ok  = df[~bad].copy()
print('RDKit 실패 / 원자·결합 0개 :', int(bad.sum()))
print('사용 가능                  :', len(ok))

dup  = ok['key14'].duplicated(keep=False)
conf = (ok[dup].groupby('key14')['Y'].nunique() > 1)
print('\nInChIKey14 unique :', ok['key14'].nunique())
print('중복에 걸린 행    :', int(dup.sum()))
print('라벨 충돌 key     :', int(conf.sum()))

pool = (ok[~ok['key14'].isin(conf[conf].index)]
        .drop_duplicates('key14', keep='first')
        .reset_index(drop=True))

print('\n=== 최종 풀 ===')
print(len(pool), pool['Y'].value_counts().to_dict())
print('분자 크기:', pool['n_atoms'].describe()[['min','50%','max']].to_dict())
print('\n[원고 Table 1 대조] 원본 %d / 사용가능 %d / 충돌key %d / 풀 %d'
      % (len(df), len(ok), int(conf.sum()), len(pool)))

RDKit 실패 / 원자·결합 0개 : 0
사용 가능                  : 13445

InChIKey14 unique : 11120
중복에 걸린 행    : 4308
라벨 충돌 key     : 83

=== 최종 풀 ===
11037 {0: 5610, 1: 5427}
분자 크기: {'min': 4.0, '50%': 31.0, 'max': 58.0}

[원고 Table 1 대조] 원본 13445 / 사용가능 13445 / 충돌key 83 / 풀 11037


In [20]:
# SMILES → key14 매핑을 재사용 (canon 재계산 안 함)
smi2key = dict(zip(df['Drug'], df['key14']))
pos     = {k: i for i, k in enumerate(pool['key14'])}

keys_by_split = {name: [smi2key.get(s) for s in part['Drug']]
                 for name, part in split.items()}

# 원래 TDC split 이 분자 수준에서 겹치는지 먼저 확인
seen, cross = {}, 0
for name in ['train','valid','test']:
    for k in keys_by_split[name]:
        if k is None: continue
        if k in seen and seen[k] != name: cross += 1
        seen.setdefault(k, name)
print('TDC 기본 split 의 분자 수준 교차 오염 :', cross, '건')
print('  → 같은 InChIKey14 가 서로 다른 split 에 들어가 있던 횟수 (리뷰어 2-5)')

# test > valid > train 우선순위로 중복 없이 배정 (평가셋을 먼저 확보)
assigned = {}
for name in ['test', 'valid', 'train']:
    for k in keys_by_split[name]:
        if k is None or k not in pos or k in assigned: continue
        assigned[k] = name

split_idx = {name: sorted(pos[k] for k, v in assigned.items() if v == name)
             for name in ['train','valid','test']}
sets  = [set(v) for v in split_idx.values()]
n_sum = sum(len(v) for v in split_idx.values())

print('\n재배정 후 :', {k: len(v) for k, v in split_idx.items()},
      '| 합계', n_sum, '/ 풀', len(pool))
assert n_sum == len(set().union(*sets)), '여전히 겹침'
print('겹침 없음 확인 ✔')

unassigned = sorted(set(range(len(pool))) - set().union(*sets))
print('미배정 분자 :', len(unassigned))

TDC 기본 split 의 분자 수준 교차 오염 : 1049 건
  → 같은 InChIKey14 가 서로 다른 split 에 들어가 있던 횟수 (리뷰어 2-5)

재배정 후 : {'train': 7309, 'valid': 1182, 'test': 2546} | 합계 11037 / 풀 11037
겹침 없음 확인 ✔
미배정 분자 : 0


In [21]:
pool[['Drug_ID','Drug','canon_smiles','key14','Y','n_atoms']].to_csv(
    OUT + 'herg_karim_pool.csv', index=False)

json.dump({**split_idx, '_unassigned': unassigned},
          open(OUT + 'herg_karim_splits.json','w'))

df.to_csv(OUT + 'herg_karim_raw_full.csv', index=False)

meta = {
    'dataset'      : 'hERG_Karim (TDC single_pred.Tox)',
    'pytdc_version': PYTDC_VERSION,
    'download_date': datetime.date.today().isoformat(),
    'raw_file'     : os.path.basename(RAW),
    'raw_md5'      : (hashlib.md5(open(RAW,'rb').read()).hexdigest()
                      if os.path.exists(RAW) else None),
    'n_raw'                : int(len(df)),
    'n_rdkit_ok'           : int(len(ok)),
    'n_label_conflict_keys': int(conf.sum()),
    'n_final'              : int(len(pool)),
    'n_duplicate_rows_removed': int(len(df) - len(pool)),
    'label_counts_final'   : {str(k): int(v) for k, v in pool['Y'].value_counts().items()},
    'split': {
        'source'   : 'TDC get_split() 기본값 (random, seed 42, frac [0.7, 0.1, 0.2])',
        'policy'   : 'InChIKey14 기준 재배정, 우선순위 test > valid > train',
        'n'        : {k: len(v) for k, v in split_idx.items()},
        'n_unassigned': len(unassigned),
        'tdc_cross_split_contamination': int(cross),
    },
    'dedup': 'InChIKey14 기준, 라벨 충돌 key 제거 후 첫 등장 유지',
}
json.dump(meta, open(OUT + 'herg_karim_provenance.json','w'), indent=2, ensure_ascii=False)
print(json.dumps(meta, indent=2, ensure_ascii=False))

{
  "dataset": "hERG_Karim (TDC single_pred.Tox)",
  "pytdc_version": "1.1.15",
  "download_date": "2026-11-12",
  "raw_file": "herg_karim.tab",
  "raw_md5": "cf699546d80721c9a160f4a6ac9b368b",
  "n_raw": 13445,
  "n_rdkit_ok": 13445,
  "n_label_conflict_keys": 83,
  "n_final": 11037,
  "n_duplicate_rows_removed": 2408,
  "label_counts_final": {
    "0": 5610,
    "1": 5427
  },
  "split": {
    "source": "TDC get_split() 기본값 (random, seed 42, frac [0.7, 0.1, 0.2])",
    "policy": "InChIKey14 기준 재배정, 우선순위 test > valid > train",
    "n": {
      "train": 7309,
      "valid": 1182,
      "test": 2546
    },
    "n_unassigned": 0,
    "tdc_cross_split_contamination": 1049
  },
  "dedup": "InChIKey14 기준, 라벨 충돌 key 제거 후 첫 등장 유지"
}
